# GPT-500M Training — Kaggle 2×T4 DDP Edition

**Architecture:** Decoder-only GPT, ~505M params (28 layers × 1152 embd × 16 heads)  
**Data:** `open-web-math/open-web-math` streamed from HuggingFace  
**Target:** Kaggle 2×T4 (2 × 15.6 GB VRAM) via DistributedDataParallel  

### How DDP works here
- `torchrun` spawns 2 worker processes (one per GPU) by running `train_ddp.py`  
- Each process owns one GPU and processes its own micro-batches independently  
- Gradients are averaged across both GPUs via NCCL all-reduce before each optimizer step  
- Effective batch: 2 GPUs × micro_batch=8 × grad_accum=8 × 1024 = **131,072 tokens/step**  
- Only rank 0 saves checkpoints and prints eval results  

### Checkpoint compatibility
Checkpoints are cross-compatible with the single-GPU Colab notebook.  
The DDP `module.` prefix is stripped on save, so `.pt` files load cleanly on either platform.

### VRAM fixes (T4 OOM)
- **8-bit AdamW** (`bitsandbytes`) — cuts optimizer state from ~4GB to ~1GB; falls back to fp32 AdamW if not installed
- **micro_batch 16→8, grad_accum 64→8** — same effective batch, lower per-step peak
- **SDPA backend pinned** to memory-efficient attention — T4 has no flash-attn-v2 kernel and can otherwise silently fall back to the much heavier "math" backend
- **`empty_cache()`** after optimizer-state load, after DDP wrap, and every 50 steps — fights allocator fragmentation over long runs
- **`max_split_size_mb:128`** added to `PYTORCH_CUDA_ALLOC_CONF`

### Checkpoint persistence — Kaggle Model registry push
Every `PUSH_EVERY_N_EVALS`-th checkpoint save pushes a copy to a Kaggle Model as a new version, so it survives independently of `/kaggle/working` and doesn't depend on committing/downloading a 5.7GB file from a live session. One-time setup is in Section 3 (Kaggle API secrets + `KAGGLE_MODEL_HANDLE`).

**Push fix (v3):** staging previously ran synchronously on the main training thread via `tempfile.mkdtemp()` (RAM-backed `/tmp` on Kaggle) — a multi-GB `shutil.copy` there could stall training for a very long time under memory pressure, which is why some steps logged eval intervals of 4000+ seconds instead of ~70s. Staging now happens entirely inside the background push thread, under `/kaggle/working` (same filesystem as `CKPT_DIR`) using `os.link()` hardlinks (falling back to `shutil.copy` only cross-device), so it never blocks the training loop and no longer competes with training for host RAM. Push is also throttled to every `PUSH_EVERY_N_EVALS` checkpoints (default 2) to conserve Kaggle Model storage quota, and failed pushes now print the full, untruncated stderr/stdout instead of the last 800 characters.

### Telemetry
Every eval interval now also writes a `training_stats.csv` row (loss/perplexity, LR, grad/weight norm, per-GPU utilization + VRAM via NVML, MFU, tokens/sec) — the same schema the Colab notebook logs to Drive, adapted for two ranks. The CSV rides along in the same Kaggle Model version as its checkpoint, and Section 3 seeds it back in on resume so history isn't lost across sessions.

### Push preflight check (no more silent hangs)
Startup now runs one connectivity/credentials check — `kaggle` CLI present, `KAGGLE_USERNAME`/`KAGGLE_KEY` set, kaggle.com reachable — instead of discovering a missing piece only after a subprocess sits for up to an hour on the first checkpoint. If the check fails, push is disabled for the session with a clear one-line reason and checkpoints/telemetry stay local-only; nothing blocks training. Outstanding uploads are also joined before the process exits, so the *last* checkpoint of a session isn't dropped mid-upload.

### Flow
1. Install → 2. Check GPUs → 3. Seed checkpoint + configure registry push → 4. Write script → 5. Verify → 6. Launch  
Re-running cell 6 resumes automatically from the latest checkpoint.

---
### Parameter accounting
```
Token embedding    :  50257 × 1152  =   57.9M  (tied with lm_head)
Position embedding :   1024 × 1152  =    1.2M
Per transformer block (×28):
  CausalSelfAttn  :  4 × 1152²     =    5.31M
  FeedForward     :  8 × 1152²     =   10.62M
  LayerNorms ×2   :  4 × 1152      =    4608
28 blocks total                     =  447.0M
Final LayerNorm                     =    2304
──────────────────────────────────────────────
TOTAL                               = ~505.1M
```


## 1. Install Dependencies

In [4]:
!pip install -q datasets transformers tokenizers tiktoken bitsandbytes pynvml kaggle


## 2. Environment Check

In [5]:
import torch

n_gpus = torch.cuda.device_count()
print(f"GPUs available : {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB")
print(f"PyTorch        : {torch.__version__}")

assert n_gpus >= 2, (
    "This notebook requires 2 GPUs. "
    "Select 'GPU T4 x2' in Kaggle accelerator settings."
)

GPUs available : 2
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB
PyTorch        : 2.10.0+cu128


## 3. Configure Paths & Seed Checkpoint

In [6]:
import os, shutil, glob

CKPT_DIR  = '/kaggle/working/GPT500M/Checkpoints'
CACHE_DIR = '/tmp/HF_Cache'   # ephemeral — fine for streaming, saves working space
SCRIPT    = '/kaggle/working/train_ddp.py'

# Guard rail: CKPT_DIR must be local scratch space (/kaggle/working), never the
# auto-mounted, READ-ONLY /kaggle/input path of an attached Model/Dataset.
# KAGGLE_MODEL_HANDLE (below) is the registry push *target* — it has nothing
# to do with this path. If you're seeing EROFS / "Read-only file system"
# errors, this is almost always the cause: CKPT_DIR got pointed at
# /kaggle/input/... instead of left under /kaggle/working.
assert not CKPT_DIR.startswith("/kaggle/input"), (
    f"CKPT_DIR is set to {CKPT_DIR!r}, which is under /kaggle/input and is "
    "READ-ONLY. Set CKPT_DIR back to a path under /kaggle/working — Kaggle "
    "mounts your attached Model there automatically; you seed FROM it, you "
    "never write TO it."
)

os.makedirs(CKPT_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)


def _safe_copy(src, dst, label):
    """shutil.copy wrapper that turns a raw EROFS traceback into an
    actionable message pointing at the most likely cause (dst accidentally
    pointing at a read-only /kaggle/input mount)."""
    try:
        shutil.copy(src, dst)
    except OSError as e:
        if e.errno == 30:  # EROFS
            raise OSError(
                f"Could not write {label} to {dst!r} — read-only file "
                f"system. dst should be under /kaggle/working; check that "
                f"CKPT_DIR / STATS_DST weren't accidentally set to an "
                f"/kaggle/input path."
            ) from e
        raise

# ── Seed checkpoint from Kaggle Model registry (first session only) ───────────
existing = glob.glob(os.path.join(CKPT_DIR, 'ckpt_*.pt'))
if not existing:
    uploaded = glob.glob('/kaggle/input/**/ckpt_*.pt', recursive=True)
    if uploaded:
        src = sorted(uploaded)[-1]
        dst = os.path.join(CKPT_DIR, os.path.basename(src))
        print(f"Seeding checkpoint: {src} -> {dst}")
        _safe_copy(src, dst, label="checkpoint")
        print("Done. Training will resume from this checkpoint.")
    else:
        print("No uploaded checkpoint found — will start from scratch.")
else:
    print(f"Checkpoint already present: {sorted(existing)[-1]}")

# ── Seed telemetry history from Kaggle Model registry (first session only) ────
# train_ddp.py bundles training_stats.csv into every checkpoint push (see
# push_checkpoint_async), so it lives alongside the ckpt_*.pt files in the
# same Model version. Seed it the same way, so stats keep accumulating across
# sessions instead of resetting to an empty CSV every time the kernel restarts.
STATS_DST = os.path.join(os.path.dirname(CKPT_DIR), "training_stats.csv")
if not os.path.isfile(STATS_DST):
    uploaded_stats = glob.glob('/kaggle/input/**/training_stats.csv', recursive=True)
    if uploaded_stats:
        src = sorted(uploaded_stats)[-1]
        print(f"Seeding telemetry history: {src} -> {STATS_DST}")
        _safe_copy(src, STATS_DST, label="telemetry CSV")
    else:
        print("No prior training_stats.csv found — will start a fresh telemetry log.")
else:
    print(f"Telemetry log already present: {STATS_DST}")

print(f"\nCheckpoint dir : {CKPT_DIR}")
print(f"HF cache dir   : {CACHE_DIR}")

# ── Kaggle Model registry push target (checkpoint persistence) ────────────────
# Every checkpoint save will also push a copy to this Kaggle Model as a new
# version, so it survives independently of /kaggle/working and this session.
#
# One-time setup before this will work:
#   1. Create the model container once, from your own machine or the Kaggle
#      UI:  kaggle models create -p <empty_dir_with_a_model-metadata.json>
#      (or just click "New Model" on kaggle.com/models and note the slug).
#   2. Add your Kaggle API credentials as Kaggle Notebook secrets named
#      KAGGLE_USERNAME and KAGGLE_KEY (Add-ons -> Secrets). These are the
#      same values from the kaggle.json you'd download from
#      kaggle.com/settings -> Create New Token.
#   3. Set KAGGLE_MODEL_HANDLE below to "<your-username>/<model-slug>/pytorch/checkpoints"
#      (the /pytorch/checkpoints part is the framework/variation slug Kaggle
#      expects — adjust if you named your variation something else).
KAGGLE_MODEL_HANDLE = "xxx"  # <-- EDIT ME

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"]      = _secrets.get_secret("KAGGLE_KEY")
    os.environ["KAGGLE_MODEL_HANDLE"] = KAGGLE_MODEL_HANDLE
    print(f"Kaggle API credentials loaded. Checkpoints will be pushed to: {KAGGLE_MODEL_HANDLE}")
except Exception as e:
    os.environ["KAGGLE_MODEL_HANDLE"] = ""
    print(f"WARNING: Kaggle API secrets not found ({e}).")
    print("Checkpoints will stay local only (no registry push) until you add")
    print("KAGGLE_USERNAME / KAGGLE_KEY as Notebook secrets.")


Checkpoint already present: /kaggle/working/GPT500M/Checkpoints/ckpt_001100.pt
Telemetry log already present: /kaggle/working/GPT500M/training_stats.csv

Checkpoint dir : /kaggle/working/GPT500M/Checkpoints
HF cache dir   : /tmp/HF_Cache
Kaggle API credentials loaded. Checkpoints will be pushed to: xxx


## 4. Write Training Script

The full training logic lives in `train_ddp.py`.  
`torchrun` will launch one copy per GPU — each knows its `LOCAL_RANK` via environment variables.

In [7]:
_script = """import contextlib
import csv
import os, gc, glob, math, socket, time, subprocess, tempfile, shutil, threading
from dataclasses import dataclass
from typing import Optional, Iterator

try:
    import pynvml
    _NVML_OK = True
except ImportError:
    _NVML_OK = False

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as grad_ckpt
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

# ── SDPA backend pin ────────────────────────────────────────────────────────
# T4 (Turing, sm_75) has no flash-attention-v2 kernel. Left to its own
# devices, PyTorch's SDPA dispatcher can silently fall back to the naive
# "math" backend, which materializes a full TxT attention matrix per head
# and burns far more VRAM. Force the memory-efficient backend explicitly.
try:
    from torch.nn.attention import sdpa_kernel, SDPBackend
    def _sdpa_ctx():
        return sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION)
except ImportError:  # older torch versions
    @contextlib.contextmanager
    def _sdpa_ctx():
        with torch.backends.cuda.sdp_kernel(
            enable_flash=False, enable_math=False, enable_mem_efficient=True
        ):
            yield

# ── DDP initialisation ────────────────────────────────────────────────────────
# IMPORTANT: allocator config + device selection must happen BEFORE
# init_process_group. NCCL init creates each process's CUDA context, and the
# caching allocator reads PYTORCH_CUDA_ALLOC_CONF lazily on first CUDA call —
# setting it after init_process_group is too late, it's silently ignored.
LOCAL_RANK = int(os.environ["LOCAL_RANK"])
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.set_device(LOCAL_RANK)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

dist.init_process_group(backend="nccl")
WORLD_SIZE = dist.get_world_size()
IS_MASTER  = (LOCAL_RANK == 0)
device = f"cuda:{LOCAL_RANK}"

if IS_MASTER:
    print(f"DDP ready | world_size={WORLD_SIZE}")
    for i in range(WORLD_SIZE):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  {mem:.1f} GB")

# NVML gives per-GPU utilization %, which torch itself doesn't expose for
# devices other than the current one. Only rank 0 needs it — it can see both
# GPUs on the box via nvmlDeviceGetHandleByIndex regardless of which GPU it
# owns. Guarded so a driver/permissions hiccup never interrupts training.
if IS_MASTER and _NVML_OK:
    try:
        pynvml.nvmlInit()
    except Exception as _e:
        _NVML_OK = False
        print(f"NVML init failed ({_e}); gpu_util columns will log as None.")
elif IS_MASTER and not _NVML_OK:
    print("pynvml not installed; gpu_util columns will log as None. "
          "Add 'pynvml' to the pip install cell to enable.")

# ── Paths ─────────────────────────────────────────────────────────────────────
CKPT_DIR  = "/kaggle/working/GPT500M/Checkpoints"
CACHE_DIR = "/tmp/HF_Cache"

# ── Kaggle Model registry push target ───────────────────────────────────────
# Set by the notebook (Section 3) via env var so checkpoints survive past this
# session without relying on /kaggle/working disk or the notebook "commit".
KAGGLE_MODEL_HANDLE = os.environ.get("KAGGLE_MODEL_HANDLE", "")

# ── Kaggle push preflight check ─────────────────────────────────────────────
# Runs ONCE at startup rather than inside every push. Without this,
# push_checkpoint_async would blindly shell out to the `kaggle` CLI on every
# checkpoint and only discover missing credentials / no CLI / no internet
# after a slow subprocess timeout — up to 3600s wasted per checkpoint, every
# checkpoint, for the whole run. Fail fast once, print exactly why, and skip
# every subsequent push cheaply instead.
def _check_kaggle_push_available():
    if not KAGGLE_MODEL_HANDLE:
        return False, "KAGGLE_MODEL_HANDLE not set"
    if shutil.which("kaggle") is None:
        return False, "'kaggle' CLI not found (add 'kaggle' to the pip install cell)"
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        return False, "KAGGLE_USERNAME / KAGGLE_KEY not set (add as Notebook secrets)"
    try:
        socket.create_connection(("www.kaggle.com", 443), timeout=5).close()
    except OSError as e:
        return False, f"no internet reachable ({e}) — enable Internet in Notebook settings"
    return True, "ok"

_PUSH_OK, _PUSH_REASON = (False, "not checked (non-master rank)")
if IS_MASTER:
    _PUSH_OK, _PUSH_REASON = _check_kaggle_push_available()
    if _PUSH_OK:
        print(f"Kaggle Model push: enabled -> {KAGGLE_MODEL_HANDLE}")
    else:
        print(f"Kaggle Model push: DISABLED ({_PUSH_REASON}). "
              f"Checkpoints will stay local-only in {CKPT_DIR} for this session.")

# Timeout per push attempt. Async, so it never blocks training — this just
# bounds how long a single stuck upload can occupy its background thread and
# how long we're willing to wait for it when joining at exit (see bottom).
_PUSH_TIMEOUT_S = 900

# Every push thread we start gets tracked here so we can join() them before
# the process exits — otherwise the LAST checkpoint of a session can be
# silently dropped mid-upload when the daemon thread is killed on exit.
_push_threads = []

# ── Telemetry (Core Model Performance / Optimization Health /
#    System & Hardware Utilization / Data Throughput) ───────────────────────
# Same schema as the Colab notebook's CSV, adapted for 2 ranks:
#  - written by rank 0 only, at eval_interval (same cadence as checkpoints)
#  - VRAM/util logged per-GPU (gpu0/gpu1 columns) since OOM can hit either
#    T4 independently and a single combined number would hide that
#  - MFU's peak-FLOPs denominator is scaled by WORLD_SIZE so it's comparable
#    to the single-GPU Colab MFU number (% of *all* attached T4s' peak)
LOCAL_STATS_CSV = os.path.join(os.path.dirname(CKPT_DIR), "training_stats.csv")
STATS_FIELDNAMES = [
    "step", "epoch",
    "train_loss", "train_perplexity", "val_loss", "val_perplexity",
    "learning_rate", "grad_norm", "weight_norm",
    "gpu0_util_pct", "gpu1_util_pct",
    "vram_gpu0_gb", "vram_gpu1_gb",
    "mfu_percentage", "step_time_s", "tokens_per_sec",
]
T4_PEAK_FLOPS_TOTAL = 65e12 * WORLD_SIZE


def _safe_perplexity(loss: float) -> float:
    try:
        return math.exp(min(loss, 20.0))
    except (OverflowError, ValueError):
        return float("inf")


def _weight_norm(model) -> float:
    with torch.no_grad():
        sq = torch.zeros((), device=device)
        for p in model.parameters():
            if p.requires_grad:
                sq += p.detach().float().pow(2).sum()
        return sq.sqrt().item()


def _gpu_snapshot():
    # Per-GPU utilization (NVML, rank 0 can see both devices) + per-GPU
    # allocated VRAM (PyTorch allocator, no NVML needed). Returns None for any
    # value that couldn't be read rather than raising.
    u0 = u1 = None
    if _NVML_OK:
        try:
            u0 = pynvml.nvmlDeviceGetUtilizationRates(
                pynvml.nvmlDeviceGetHandleByIndex(0)).gpu
        except Exception:
            pass
        if WORLD_SIZE > 1:
            try:
                u1 = pynvml.nvmlDeviceGetUtilizationRates(
                    pynvml.nvmlDeviceGetHandleByIndex(1)).gpu
            except Exception:
                pass
    v0 = torch.cuda.memory_allocated(0) / 1e9
    v1 = torch.cuda.memory_allocated(1) / 1e9 if WORLD_SIZE > 1 else None
    return u0, u1, v0, v1


def init_stats_csv():
    # Write a header only if no CSV is present yet. If the notebook's
    # seed-checkpoint cell already copied a prior session's training_stats.csv
    # into /kaggle/working (mirroring how it seeds ckpt_*.pt), this leaves it
    # alone so stats history keeps accumulating across sessions instead of
    # resetting every time the kernel restarts.
    if os.path.isfile(LOCAL_STATS_CSV):
        if IS_MASTER:
            print(f"Resuming existing stats CSV: {LOCAL_STATS_CSV}")
        return
    with open(LOCAL_STATS_CSV, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=STATS_FIELDNAMES).writeheader()
    if IS_MASTER:
        print(f"Initialized fresh stats CSV: {LOCAL_STATS_CSV}")


def write_stats_row(row: dict):
    with open(LOCAL_STATS_CSV, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=STATS_FIELDNAMES).writerow(row)


# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class GPTConfig:
    # Architecture
    vocab_size : int   = 50257
    block_size : int   = 1024
    n_layer    : int   = 28
    n_head     : int   = 16
    n_embd     : int   = 1152
    dropout    : float = 0.0

    # Training
    # 2xT4 DDP: grad_accum_steps is HALF of the Colab (1-GPU) value because
    # DDP already multiplies effective batch by WORLD_SIZE (each rank runs
    # its own accumulation, gradients are all-reduced across ranks). Keeping
    # grad_accum_steps identical to Colab silently doubles tokens/step.
    # effective batch = WORLD_SIZE(2) x micro_batch(8) x grad_accum(8) x
    #                   block_size(1024) = 131,072 tokens/step -- matches Colab.
    num_epochs        : int   = 2
    micro_batch_size  : int   = 8
    grad_accum_steps  : int   = 8
    lr                : float = 3e-4
    weight_decay      : float = 0.1
    grad_clip         : float = 1.0
    warmup_steps      : int   = 1000
    train_steps       : int   = 112000
    eval_interval     : int   = 100
    eval_batches      : int   = 50
    tokenizer_name    : str   = "gpt2"
    max_checkpoints   : int   = 3
    device            : str   = "cuda"

cfg = GPTConfig()
cfg.device = device


# ─────────────────────────────────────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────────────────────────────────────
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head   = cfg.n_head
        self.n_embd   = cfg.n_embd
        self.head_dim = cfg.n_embd // cfg.n_head
        self.dropout  = cfg.dropout
        self.c_attn     = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.c_proj     = nn.Linear(cfg.n_embd, cfg.n_embd,     bias=False)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self._cache_k: Optional[torch.Tensor] = None
        self._cache_v: Optional[torch.Tensor] = None

    def forward(self, x: torch.Tensor, use_cache: bool = False) -> torch.Tensor:
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        def split_heads(t):
            return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k, v = split_heads(q), split_heads(k), split_heads(v)
        if use_cache:
            if self._cache_k is not None:
                k = torch.cat([self._cache_k, k], dim=2)
                v = torch.cat([self._cache_v, v], dim=2)
            self._cache_k = k
            self._cache_v = v
        dropout_p = self.dropout if self.training else 0.0
        with _sdpa_ctx():
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
                                               dropout_p=dropout_p, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

    def clear_cache(self):
        self._cache_k = None
        self._cache_v = None


class FeedForward(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False),
            nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False),
            nn.Dropout(cfg.dropout),
        )
    def forward(self, x): return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg.n_embd)
        self.ffn  = FeedForward(cfg)

    def _block_fn(self, x):
        x = x + self.attn(self.ln1(x), use_cache=False)
        x = x + self.ffn(self.ln2(x))
        return x

    def forward(self, x, use_cache=False):
        if self.training and not use_cache:
            return grad_ckpt.checkpoint(self._block_fn, x, use_reentrant=False)
        x = x + self.attn(self.ln1(x), use_cache=use_cache)
        x = x + self.ffn(self.ln2(x))
        return x


class GPT500M(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg      = cfg
        self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb  = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop     = nn.Dropout(cfg.dropout)
        self.blocks   = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_final = nn.LayerNorm(cfg.n_embd)
        self.head     = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self._init_weights()
        if IS_MASTER:
            n = self.num_params()
            print(f"GPT-500M | {n:,} params ({n/1e6:.1f}M)")

    def _init_weights(self):
        for name, module in self.named_modules():
            if isinstance(module, nn.Linear):
                std = 0.02
                if name.endswith(("c_proj", "net.2")):
                    std = 0.02 / math.sqrt(2 * self.cfg.n_layer)
                nn.init.normal_(module.weight, mean=0.0, std=std)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def num_params(self):
        return sum(p.numel() for n, p in self.named_parameters() if n != "head.weight")

    def forward(self, idx, targets=None, use_cache=False):
        B, T = idx.shape
        assert T <= self.cfg.block_size
        positions = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(positions))
        for block in self.blocks:
            x = block(x, use_cache=use_cache)
        x      = self.ln_final(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def clear_kv_cache(self):
        for block in self.blocks:
            block.attn.clear_cache()


# ─────────────────────────────────────────────────────────────────────────────
# CHECKPOINT UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
# Staging lives under /kaggle/working (same filesystem as CKPT_DIR), never under
# /tmp. /tmp on Kaggle is RAM-backed (tmpfs), so staging a multi-GB checkpoint
# there competes with training for host memory under pressure. Same filesystem
# also means os.link() (an instant hardlink) works instead of a byte-for-byte
# shutil.copy — copy is kept only as a fallback for cross-device edge cases.
PUSH_STAGING_DIR = "/kaggle/working/GPT500M/_push_staging"
os.makedirs(PUSH_STAGING_DIR, exist_ok=True)

# Throttle: push only every Nth checkpoint save instead of every eval_interval.
# Each push creates a brand-new full-size Kaggle Model version, so pushing on
# every eval burns through storage quota fast when eval_interval is small.
# Set to 1 to push every checkpoint (previous behavior). Local checkpoints under
# CKPT_DIR are still saved every eval regardless of this setting.
PUSH_EVERY_N_EVALS = 2
_eval_push_count = 0


def _stage_checkpoint_files(ckpt_path, step, extra_files=None):
    # Build a per-push staging directory and populate it with hardlinks of
    # the checkpoint (+ extra files). Hardlinking means the staged copy
    # stays valid even if checkpoint rotation deletes the original
    # mid-upload — the inode isn't freed until every link to it is gone.
    tmp_dir = tempfile.mkdtemp(dir=PUSH_STAGING_DIR)
    for source_path in [ckpt_path] + [f for f in (extra_files or []) if os.path.isfile(f)]:
        dst = os.path.join(tmp_dir, os.path.basename(source_path))
        try:
            os.link(source_path, dst)
        except OSError:
            shutil.copy(source_path, dst)
    return tmp_dir


def _push_checkpoint_worker(ckpt_path, step, extra_files=None):
    # Staging (the expensive multi-GB part) now happens HERE, inside the
    # background thread — never on the main training thread — so it can never
    # stall the training loop the way the old synchronous shutil.copy did.
    try:
        tmp_dir = _stage_checkpoint_files(ckpt_path, step, extra_files)
    except Exception as e:
        print(f"  [kaggle-push] WARNING: could not stage files for upload: {e}")
        return
    try:
        result = subprocess.run(
            ["kaggle", "models", "instances", "versions", "create",
             KAGGLE_MODEL_HANDLE, "-p", tmp_dir, "-n", f"step {step}"],
            capture_output=True, text=True, timeout=_PUSH_TIMEOUT_S,
        )
        if result.returncode == 0:
            print(f"  [kaggle-push] step {step} checkpoint uploaded -> {KAGGLE_MODEL_HANDLE}")
        else:
            # Full stderr AND stdout, untruncated — a short/empty stderr was
            # previously indistinguishable from "no error info captured at all".
            print(f"  [kaggle-push] WARNING: upload failed for step {step} "
                  f"(exit code {result.returncode}):\\n"
                  f"  stderr: {result.stderr!r}\\n"
                  f"  stdout: {result.stdout!r}")
    except subprocess.TimeoutExpired:
        print(f"  [kaggle-push] WARNING: upload for step {step} exceeded "
              f"{_PUSH_TIMEOUT_S}s and was aborted — this checkpoint is local-only.")
    except Exception as e:
        print(f"  [kaggle-push] WARNING: upload failed for step {step}: {e}")
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


def push_checkpoint_async(ckpt_path, step, extra_files=None):
    global _eval_push_count
    if not _PUSH_OK:
        print(f"  [kaggle-push] skipping step {step} ({_PUSH_REASON}) — local checkpoint only.")
        return
    _eval_push_count += 1
    if (_eval_push_count % PUSH_EVERY_N_EVALS) != 0:
        print(f"  [kaggle-push] skipping step {step} (throttled — pushing every "
              f"{PUSH_EVERY_N_EVALS} checkpoint(s) to conserve quota); local checkpoint only.")
        return
    t = threading.Thread(target=_push_checkpoint_worker,
                          args=(ckpt_path, step, extra_files), daemon=True)
    t.start()
    _push_threads.append(t)


def save_checkpoint(model, optimizer, scaler, step, epoch, epoch_step, losses, cfg):
    raw_sd    = model.module.state_dict()
    ckpt_path = os.path.join(CKPT_DIR, f"ckpt_{step:06d}.pt")
    torch.save({
        "step"       : step,
        "epoch"      : epoch,
        "epoch_step" : epoch_step,
        "model"      : raw_sd,
        "optimizer"  : optimizer.state_dict(),
        "scaler"     : scaler.state_dict(),
        "cfg"        : cfg,
        "losses"     : losses,
    }, ckpt_path)

    train_loss = losses["train"]
    val_loss   = losses["val"]
    print(f"  checkpoint saved -> {ckpt_path}  "
          f"(epoch {epoch} | step {step} | train {train_loss:.4f} | val {val_loss:.4f})")

    # Push a copy to the Kaggle Model registry (async, non-blocking) so it
    # persists independently of /kaggle/working and this session. Bundle the
    # telemetry CSV in the same version so stats history always travels with
    # the checkpoint that produced it.
    push_checkpoint_async(ckpt_path, step, extra_files=[LOCAL_STATS_CSV])

    # Rotate — keep only max_checkpoints most recent
    all_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "ckpt_*.pt")),
                       key=os.path.getmtime)
    for old in all_ckpts[: max(0, len(all_ckpts) - cfg.max_checkpoints)]:
        os.remove(old)
        print(f"  deleted old checkpoint: {old}")


def load_checkpoint(path, raw_model, optimizer, scaler):
    global cfg
    print(f"Loading checkpoint: {path}")
    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    # Only pull ARCHITECTURE fields from the checkpoint's saved cfg -- these
    # must match the saved weights' shapes for load_state_dict to work.
    # Deliberately do NOT overwrite TRAINING fields (micro_batch_size,
    # grad_accum_steps, lr, num_epochs, ...) with whatever was in effect when
    # the checkpoint was saved -- the module-level `cfg` already reflects the
    # CURRENT script's intended hyperparameters and should win. Previously
    # this line did `cfg = ckpt["cfg"]`, wholesale-replacing every field,
    # which silently reverted grad_accum_steps (and any other training
    # hyperparameter) back to whatever it was when the checkpoint was saved,
    # on every single resume.
    ckpt_cfg = ckpt["cfg"]
    for _field in ("vocab_size", "block_size", "n_layer", "n_head", "n_embd", "dropout"):
        if hasattr(ckpt_cfg, _field):
            setattr(cfg, _field, getattr(ckpt_cfg, _field))
    cfg.device = device

    # Strip module. prefix (safety net — save_checkpoint already does this)
    sd = {k.replace("module.", ""): v for k, v in ckpt["model"].items()}
    raw_model.load_state_dict(sd)
    del sd, ckpt["model"]

    optimizer.load_state_dict(ckpt["optimizer"])
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(device)
    del ckpt["optimizer"]

    if "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])

    gc.collect()
    torch.cuda.empty_cache()

    step       = ckpt["step"]
    epoch      = ckpt.get("epoch",      0)
    epoch_step = ckpt.get("epoch_step", 0)
    losses     = ckpt["losses"]

    if IS_MASTER:
        train_loss = losses["train"]
        val_loss   = losses["val"]
        allocated  = torch.cuda.memory_allocated() / 1e9
        reserved   = torch.cuda.memory_reserved()  / 1e9
        print(f"  resumed — global step {step} | epoch {epoch} | epoch_step {epoch_step}")
        print(f"  train loss {train_loss:.4f} | val loss {val_loss:.4f}")
        print(f"  VRAM after load: {allocated:.2f} GB allocated / {reserved:.2f} GB reserved")

    return cfg, step, epoch, epoch_step, losses


# ─────────────────────────────────────────────────────────────────────────────
# DATA PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
import tiktoken
from datasets import load_dataset

enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token


class TokenBuffer:
    def __init__(self, iterator, block_size: int):
        self._iter       = iterator
        self._block_size = block_size
        self._buf: list  = []

    def _generate(self) -> Iterator:
        for doc in self._iter:
            tokens = enc.encode_ordinary(doc["text"]) + [EOT]
            self._buf.extend(tokens)
            while len(self._buf) >= self._block_size + 1:
                chunk = self._buf[: self._block_size + 1]
                self._buf = self._buf[self._block_size + 1 :]
                yield chunk


def make_stream(epoch: int, skip_docs: int = 0):
    ds = load_dataset(
        "open-web-math/open-web-math",
        split     = "train",
        streaming = True,
        cache_dir = CACHE_DIR,
    )
    # Each rank shuffles with a different seed so they see different docs
    ds = ds.shuffle(seed=42 + epoch + LOCAL_RANK * 1000, buffer_size=10_000)
    if skip_docs > 0 and IS_MASTER:
        print(f"  fast-forwarding {skip_docs:,} documents in epoch {epoch} ...")
    if skip_docs > 0:
        ds = ds.skip(skip_docs)
    return iter(ds)


def make_val_tensors(n_val_docs: int = 500, block_size: int = 1024):
    if IS_MASTER:
        print(f"Building validation set from {n_val_docs} val docs ...")
    val_ds = load_dataset(
        "open-web-math/open-web-math",
        split     = "train",
        streaming = True,
        cache_dir = CACHE_DIR,
    )
    val_tokens = []
    for i, doc in enumerate(val_ds):
        if i >= n_val_docs:
            break
        val_tokens.extend(enc.encode_ordinary(doc["text"]) + [EOT])
    n = (len(val_tokens) // (block_size + 1)) * (block_size + 1)
    return torch.tensor(val_tokens[:n], dtype=torch.int64)


# ─────────────────────────────────────────────────────────────────────────────
# TRAINING UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
def get_lr(step: int, cfg: GPTConfig) -> float:
    if step < cfg.warmup_steps:
        return cfg.lr * (step + 1) / cfg.warmup_steps
    progress = (step - cfg.warmup_steps) / max(1, cfg.train_steps - cfg.warmup_steps)
    return cfg.lr * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))


@torch.no_grad()
def estimate_loss(model, val_data, cfg):
    loss_tensor = torch.zeros(1, device=device)
    if IS_MASTER:
        model.eval()
        val_gpu   = val_data.to(device)
        max_start = len(val_gpu) - cfg.block_size - 1
        losses = []
        with torch.amp.autocast("cuda"):
            for _ in range(cfg.eval_batches):
                ix = torch.randint(max_start, (cfg.micro_batch_size,))
                x  = torch.stack([val_gpu[i     : i + cfg.block_size    ] for i in ix])
                y  = torch.stack([val_gpu[i + 1 : i + cfg.block_size + 1] for i in ix])
                _, loss = model(x, y)
                losses.append(loss.item())
        del val_gpu
        torch.cuda.empty_cache()
        loss_tensor[0] = sum(losses) / len(losses)
        model.train()
    dist.broadcast(loss_tensor, src=0)
    return {"val": loss_tensor.item()}


# ─────────────────────────────────────────────────────────────────────────────
# BUILD MODEL, OPTIMIZER, SCALER
# ─────────────────────────────────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

raw_model = GPT500M(cfg)

decay_params    = [p for n, p in raw_model.named_parameters() if p.dim() >= 2 and p.requires_grad]
no_decay_params = [p for n, p in raw_model.named_parameters() if p.dim() <  2 and p.requires_grad]

# 8-bit optimizer state cuts AdamW's exp_avg/exp_avg_sq footprint from ~4GB to
# ~1GB on a 505M-param model. Prefer the *paged* variant: it spills state to
# CPU pinned memory under GPU pressure instead of raising OOM, which gives a
# real safety margin on a card this close to the edge. Falls back to plain
# AdamW8bit, then to standard fp32 AdamW if bitsandbytes isn't installed.
try:
    import bitsandbytes as bnb
    OptClass = getattr(bnb.optim, "PagedAdamW8bit", None) or bnb.optim.AdamW8bit
    optimizer = OptClass([
        {"params": decay_params,    "weight_decay": cfg.weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ], lr=cfg.lr, betas=(0.9, 0.95))
    if IS_MASTER:
        print(f"Optimizer: bitsandbytes {OptClass.__name__} (reduced VRAM optimizer state)")
except ImportError:
    optimizer = torch.optim.AdamW([
        {"params": decay_params,    "weight_decay": cfg.weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ], lr=cfg.lr, betas=(0.9, 0.95), fused=True)
    if IS_MASTER:
        print("Optimizer: fp32 AdamW (bitsandbytes not found — pip install bitsandbytes to save VRAM)")

scaler = torch.amp.GradScaler("cuda")

def _log_vram(label):
    # Per-rank VRAM snapshot — printed from every rank, since OOM can hit
    # either GPU independently and rank-0-only logging hides that.
    a = torch.cuda.memory_allocated(device) / 1e9
    r = torch.cuda.memory_reserved(device)  / 1e9
    print(f"  [vram][rank{LOCAL_RANK}] {label}: allocated {a:.2f}GB reserved {r:.2f}GB")

# ── Resume or fresh start ─────────────────────────────────────────────────────
checkpoints = sorted(glob.glob(os.path.join(CKPT_DIR, "ckpt_*.pt")), key=os.path.getmtime)
latest_ckpt = checkpoints[-1] if checkpoints else None

raw_model.to(device)
_log_vram("after model.to(device)")

if latest_ckpt:
    cfg, start_step, start_epoch, start_epoch_step, loaded_losses = \
        load_checkpoint(latest_ckpt, raw_model, optimizer, scaler)
else:
    if IS_MASTER:
        print("No checkpoint found — starting from scratch.")
    start_step       = 0
    start_epoch      = 0
    start_epoch_step = 0
    loaded_losses    = {"train": float("nan"), "val": float("nan")}
_log_vram("after checkpoint resume / fresh init")

# Wrap in DDP AFTER loading weights
model = DDP(raw_model, device_ids=[LOCAL_RANK], find_unused_parameters=False)
torch.cuda.empty_cache()
_log_vram("after DDP wrap")

# ── Step & epoch counters ─────────────────────────────────────────────────────
# STEPS_PER_EPOCH must be the number of GLOBAL optimizer steps to consume one
# epoch's worth of tokens. Each global step pulls
# WORLD_SIZE * micro_batch_size * grad_accum_steps sequences (one micro-batch
# per rank per accum step, across all ranks) — NOT just grad_accum_steps
# worth. Omitting WORLD_SIZE/micro_batch_size here undercounts tokens/step by
# that factor, which overcounts STEPS_PER_EPOCH (and therefore train_steps)
# by the same factor: on 2xT4 this ran exactly 2x too many steps per epoch.
DOCS_PER_EPOCH  = 6_300_000
STEPS_PER_EPOCH = (DOCS_PER_EPOCH * 350) // (cfg.block_size + 1) // (
    WORLD_SIZE * cfg.micro_batch_size * cfg.grad_accum_steps
)
cfg.train_steps = STEPS_PER_EPOCH * cfg.num_epochs

if IS_MASTER:
    tokens_per_step = WORLD_SIZE * cfg.micro_batch_size * cfg.grad_accum_steps * cfg.block_size
    print(f"\\nSteps per epoch       : {STEPS_PER_EPOCH:,}")
    print(f"Total steps           : {cfg.train_steps:,} ({cfg.num_epochs} epoch(s))")
    print(f"Start step            : {start_step}")
    print(f"Effective tokens/step : {tokens_per_step:,}")


# ─────────────────────────────────────────────────────────────────────────────
# BUILD VAL SET (rank 0 only) & DATA STREAM
# ─────────────────────────────────────────────────────────────────────────────
val_data = make_val_tensors(n_val_docs=500, block_size=cfg.block_size) if IS_MASTER else None
if IS_MASTER:
    print(f"Val set ready — {len(val_data):,} tokens")

train_stream_iter = make_stream(
    epoch     = start_epoch,
    skip_docs = start_epoch_step * cfg.grad_accum_steps * cfg.micro_batch_size,
)
train_buf = TokenBuffer(train_stream_iter, block_size=cfg.block_size)
buf_gen   = train_buf._generate()


# ─────────────────────────────────────────────────────────────────────────────
# TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
if IS_MASTER:
    init_stats_csv()

model.train()
train_loss_acc = 0.0
t0 = time.time()

epoch      = start_epoch
epoch_step = start_epoch_step

for step in range(start_step, cfg.train_steps):

    # Epoch boundary
    if epoch_step >= STEPS_PER_EPOCH:
        epoch      += 1
        epoch_step  = 0
        if epoch >= cfg.num_epochs:
            if IS_MASTER:
                print(f"\\nCompleted {cfg.num_epochs} epoch(s). Training done.")
            break
        if IS_MASTER:
            sep = "=" * 60
            print(f"\\n{sep}\\nStarting epoch {epoch}\\n{sep}")
        train_stream_iter = make_stream(epoch=epoch, skip_docs=0)
        train_buf = TokenBuffer(train_stream_iter, block_size=cfg.block_size)
        buf_gen   = train_buf._generate()

    # LR schedule
    lr = get_lr(step, cfg)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0

    with torch.amp.autocast("cuda"):
        for micro_step in range(cfg.grad_accum_steps):
            try:
                chunks = [next(buf_gen) for _ in range(cfg.micro_batch_size)]
            except StopIteration:
                epoch_step = STEPS_PER_EPOCH
                break

            data = torch.tensor(chunks, dtype=torch.int64)
            x = data[:, :-1].to(device)
            y = data[:, 1: ].to(device)

            # Only sync gradients on the last micro-step
            is_last_micro = (micro_step == cfg.grad_accum_steps - 1)
            ctx = model.no_sync() if not is_last_micro else contextlib.nullcontext()
            with ctx:
                _, loss = model(x, y)
                loss = loss / cfg.grad_accum_steps
                scaler.scale(loss).backward()
            accum_loss += loss.item()

    scaler.unscale_(optimizer)
    # clip_grad_norm_ already computes the global L2 grad norm to do the
    # clip — capture its return value here instead of recomputing it in the
    # telemetry block below.
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
    scaler.step(optimizer)
    scaler.update()

    epoch_step     += 1
    train_loss_acc += accum_loss

    if step == start_step:
        _log_vram("after first optimizer.step()")

    if step % 50 == 0:
        torch.cuda.empty_cache()
        if step != start_step:
            _log_vram(f"step {step}")
            
    # --------------- For time gauge ----------------- #
    # if IS_MASTER and step % 10 == 0:
    #     smooth_loss = train_loss_acc / max(1, step - start_step + 1)
    #     print(f"ep {epoch} | step {step:>6d}/{cfg.train_steps} | "
    #           f"loss {accum_loss:.4f} | smooth {smooth_loss:.4f} | lr {lr:.2e}", end="\\r")
    # --------------- For time gauge ----------------- #
    
    if (step % cfg.eval_interval == 0) or (step == cfg.train_steps - 1):
        # All ranks participate (dist.broadcast inside estimate_loss)
        val_losses   = estimate_loss(model, val_data, cfg)
        smooth_train = train_loss_acc / max(1, step - start_step + 1)
        losses       = {"train": smooth_train, "val": val_losses["val"]}

        if IS_MASTER:
            elapsed = time.time() - t0
            sep = "-" * 65
            print(f"\\n{sep}")
            print(f"EVAL  ep {epoch} | step {step:>6d} | "
                  f"train {losses['train']:.4f} | val {losses['val']:.4f} | {elapsed:.0f}s")

            step_time_s = elapsed / max(1, cfg.eval_interval)
            tokens_per_sec = tokens_per_step / step_time_s
            u0, u1, v0, v1 = _gpu_snapshot()
            mfu = 100.0 * (tokens_per_sec * 6 * raw_model.num_params()) / T4_PEAK_FLOPS_TOTAL

            write_stats_row({
                "step": step, "epoch": epoch,
                "train_loss": losses["train"], "train_perplexity": _safe_perplexity(losses["train"]),
                "val_loss": losses["val"], "val_perplexity": _safe_perplexity(losses["val"]),
                "learning_rate": lr, "grad_norm": grad_norm, "weight_norm": _weight_norm(raw_model),
                "gpu0_util_pct": u0, "gpu1_util_pct": u1,
                "vram_gpu0_gb": v0, "vram_gpu1_gb": v1,
                "mfu_percentage": mfu, "step_time_s": step_time_s, "tokens_per_sec": tokens_per_sec,
            })

            save_checkpoint(model, optimizer, scaler,
                            step, epoch, epoch_step, losses, cfg)
            t0 = time.time()

        # All ranks sync before continuing
        dist.barrier()
        model.train()

if IS_MASTER:
    print("\\nTraining complete.")
    if _push_threads:
        pending = [t for t in _push_threads if t.is_alive()]
        if pending:
            print(f"Waiting on {len(pending)} pending checkpoint upload(s) "
                  f"before exit (up to {_PUSH_TIMEOUT_S}s each)...")
        for t in _push_threads:
            t.join(timeout=_PUSH_TIMEOUT_S)
        still_running = [t for t in _push_threads if t.is_alive()]
        if still_running:
            print(f"WARNING: {len(still_running)} checkpoint upload(s) did not "
                  f"finish before exit — the final checkpoint may be local-only "
                  f"this session. It will still be seeded from /kaggle/working "
                  f"if you commit the notebook.")

dist.destroy_process_group()"""

with open(SCRIPT, 'w') as _f:
    _f.write(_script)
print(f"Training script written to {SCRIPT}")
print(f"Lines : {len(_script.splitlines())}")


Training script written to /kaggle/working/train_ddp.py
Lines : 870


## 5. Verify Script

In [8]:
import ast
with open(SCRIPT) as f:
    src = f.read()
try:
    ast.parse(src)
    print("✓ Script syntax OK")
    print(f"  Lines : {len(src.splitlines())}")
    print(f"  Path  : {SCRIPT}")
except SyntaxError as e:
    print(f"✗ Syntax error: {e}")

✓ Script syntax OK
  Lines : 870
  Path  : /kaggle/working/train_ddp.py


## 6. Launch Training

`torchrun --nproc_per_node=2` spawns 2 processes, one per GPU.  
Rank 0 handles all printing — output will not be duplicated.  

**To resume:** just re-run this cell. The script auto-detects the latest checkpoint.

In [ ]:
!torchrun --nproc_per_node=2 --master_port=29500 {SCRIPT}

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
W0710 16:42:35.223000 115 torch/distributed/run.py:852] 
W0710 16:42:35.223000 115 torch/distributed/run.py:852] *****************************************
W0710 16:42:35.223000 115 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0710 16:42:35.223000 115 torch/distributed/run.py:852] *****************************************
/kaggle/working/train_ddp.py:8: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please

## 7. Inspect Checkpoint

Loads the latest checkpoint metadata for a quick sanity check.  
Paste the model class here and uncomment the generation block to run inference.

In [ ]:
import os, glob, torch

CKPT_DIR = '/kaggle/working/GPT500M/Checkpoints'
checkpoints = sorted(glob.glob(os.path.join(CKPT_DIR, 'ckpt_*.pt')), key=os.path.getmtime)
assert checkpoints, "No checkpoints found — run the training cell first."

latest = checkpoints[-1]
print(f"Loading: {latest}")
ckpt = torch.load(latest, map_location='cpu', weights_only=False)

step       = ckpt['step']
epoch      = ckpt.get('epoch', 0)
train_loss = ckpt['losses']['train']
val_loss   = ckpt['losses']['val']
print(f"Step       : {step}")
print(f"Epoch      : {epoch}")
print(f"Train loss : {train_loss:.4f}")
print(f"Val loss   : {val_loss:.4f}")

# ── Telemetry history ───────────────────────────────────────────────────────────────────────
STATS_CSV = os.path.join(os.path.dirname(CKPT_DIR), "training_stats.csv")
if os.path.isfile(STATS_CSV):
    import pandas as pd
    stats = pd.read_csv(STATS_CSV)
    print(f"\nTelemetry rows: {len(stats)}  (source: {STATS_CSV})")
    display(stats.tail(10))
else:
    print(f"\nNo telemetry CSV found yet at {STATS_CSV}")

# ── Generation (uncomment after pasting model classes above) ─────────────────
# import tiktoken, math
# from train_ddp import GPT500M   # or paste the class here
# device    = 'cuda:0'
# enc       = tiktoken.get_encoding('gpt2')
# gen_cfg   = ckpt['cfg']
# gen_model = GPT500M(gen_cfg).to(device)
# sd = {k.replace('module.', ''): v for k, v in ckpt['model'].items()}
# gen_model.load_state_dict(sd)
# gen_model.eval()
#
# PROMPT         = "The derivative of sin(x) with respect to x is"
# MAX_NEW_TOKENS = 256
# TEMPERATURE    = 0.8
# TOP_K          = 50
#
# ids = enc.encode_ordinary(PROMPT)
# x   = torch.tensor([ids], dtype=torch.int64).to(device)
# with torch.no_grad():
#     for _ in range(MAX_NEW_TOKENS):
#         logits, _ = gen_model(x[:, -gen_cfg.block_size:])
#         logits = logits[:, -1, :] / TEMPERATURE
#         top_k_vals, top_k_idx = torch.topk(logits, TOP_K)
#         probs = torch.softmax(top_k_vals, dim=-1)
#         next_id = top_k_idx[0, torch.multinomial(probs[0], 1)]
#         x = torch.cat([x, next_id.unsqueeze(0).unsqueeze(0)], dim=1)
# print(enc.decode(x[0].tolist()))